# Pose Normalization

Per-frame normalization using shoulder center as origin and torso size as scale.

**Input:** `keypoints_interpolated_15_boundary`  
**Output:** shoulder center = (0, 0), torso size = 1

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

IN_DIR  = Path('../data/processed/keypoints_interpolated_15_boundary')
OUT_DIR = Path('../data/processed/keypoints_normalized')

MODELS = [
    ('movenet',        'confidence'),
    ('mediapipe_norm', 'visibility'),
]

JOINTS = [
    'left_shoulder', 'right_shoulder',
    'left_elbow',    'right_elbow',
    'left_wrist',    'right_wrist',
    'left_hip',      'right_hip',
    'left_knee',     'right_knee',
    'left_ankle',    'right_ankle',
]

## Normalization function

Frames with missing reference joints (shoulder or hip) are left as NaN.

In [ ]:
def normalize_pose(df):
    df = df.copy()

    sc_x = (df['left_shoulder_x'] + df['right_shoulder_x']) / 2
    sc_y = (df['left_shoulder_y'] + df['right_shoulder_y']) / 2

    hc_x = (df['left_hip_x'] + df['right_hip_x']) / 2
    hc_y = (df['left_hip_y'] + df['right_hip_y']) / 2

    torso_size = np.sqrt((sc_x - hc_x) ** 2 + (sc_y - hc_y) ** 2)
    invalid = (torso_size == 0) | torso_size.isna()

    for joint in JOINTS:
        df[f'{joint}_x'] = (df[f'{joint}_x'] - sc_x) / torso_size
        df[f'{joint}_y'] = (df[f'{joint}_y'] - sc_y) / torso_size
        df.loc[invalid, f'{joint}_x'] = np.nan
        df.loc[invalid, f'{joint}_y'] = np.nan

    return df

In [7]:
for model, _ in MODELS:
    out_model = OUT_DIR / model
    out_model.mkdir(parents=True, exist_ok=True)
    print(f'=== {model} ===')

    for csv_path in sorted((IN_DIR / model).glob('*.csv')):
        df      = pd.read_csv(csv_path)
        df_norm = normalize_pose(df)
        df_norm.to_csv(out_model / csv_path.name, index=False)
        print(f'  {csv_path.stem}: {len(df)} frames')
    print()

=== movenet ===
  DJI_20250425092743_0028_D_movenet: 94 frames
  DJI_20250425093100_0030_D_movenet: 92 frames
  DJI_20250425104507_0045_D_movenet: 73 frames
  DJI_20250425104804_0047_D_movenet: 94 frames
  DJI_20250425112502_0059_D_movenet: 123 frames
  DJI_20250425112749_0061_D_movenet: 151 frames
  DJI_20250425120835_0074_D_movenet: 60 frames
  DJI_20250425121226_0076_D_movenet: 151 frames
  DJI_20250425125202_0091_D_movenet: 61 frames
  DJI_20250425125448_0093_D_movenet: 91 frames

=== mediapipe_norm ===
  DJI_20250425092743_0028_D_mediapipe_norm: 94 frames
  DJI_20250425093100_0030_D_mediapipe_norm: 92 frames
  DJI_20250425104507_0045_D_mediapipe_norm: 73 frames
  DJI_20250425104804_0047_D_mediapipe_norm: 94 frames
  DJI_20250425112502_0059_D_mediapipe_norm: 123 frames
  DJI_20250425112749_0061_D_mediapipe_norm: 151 frames
  DJI_20250425120835_0074_D_mediapipe_norm: 60 frames
  DJI_20250425121226_0076_D_mediapipe_norm: 151 frames
  DJI_20250425125202_0091_D_mediapipe_norm: 61 frame

## Verify

Shoulder center should be (0, 0) in every normalized frame.

In [8]:
for model, _ in MODELS:
    csv_path = sorted((OUT_DIR / model).glob('*.csv'))[0]
    df = pd.read_csv(csv_path).dropna(subset=['left_shoulder_x'])

    sc_x = (df['left_shoulder_x'] + df['right_shoulder_x']) / 2
    sc_y = (df['left_shoulder_y'] + df['right_shoulder_y']) / 2

    print(f'{model}: {csv_path.stem}')
    print(f'  shoulder center x — mean: {sc_x.mean():.6f}, max abs: {sc_x.abs().max():.6f}')
    print(f'  shoulder center y — mean: {sc_y.mean():.6f}, max abs: {sc_y.abs().max():.6f}')
    print()

movenet: DJI_20250425092743_0028_D_movenet
  shoulder center x — mean: 0.000000, max abs: 0.000000
  shoulder center y — mean: 0.000000, max abs: 0.000000

mediapipe_norm: DJI_20250425092743_0028_D_mediapipe_norm
  shoulder center x — mean: -0.000000, max abs: 0.000000
  shoulder center y — mean: -0.000000, max abs: 0.000000

